# 02 · Agent Graph와 Tool Calling
01의 고정 Agent Graph를 그대로 실행하면서, 각 전문 Agent가 필요할 때만 할당된 읽기 전용 Tool을 호출합니다.

**학습 목표:** 공개 코어 API `run_food_mcp_loop()`로 기존 Graph의 Tool Calling 결과와 trace를 확인합니다.  
**실행 경계:** 별도 MCP 서버나 별도 planner loop는 사용하지 않으며, 읽기 전용 Tool만 제공하고 외부 쓰기 작업은 실행하지 않습니다.  
**환경 변수:** 실제 AI Core LLM을 사용하므로 루트 `.env` 설정이 필요합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: 01과 같은 Agent Graph + Tool Calling</h2>
<p style="margin: 7px 0;">01과 동일한 대화창·상태 카드·세로 타임라인을 사용합니다. Tool이 실제로 선택되면 해당 Agent 카드에서 종류, 입력, 읽기 전용 결과를 펼쳐 확인합니다.</p>
  </div>
</div>


In [1]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))

from src.loop_ui import launch_loop_ui
loop_ui = launch_loop_ui()


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [3]:
# Marketing HTML Agent가 생성한 문서 미리보기
from html import escape
from IPython.display import HTML, display
from src.food_tools import latest_marketing_html_draft

latest_ui_draft = latest_marketing_html_draft()
graph_tool_calls = result.get('tool_calls', []) if 'result' in globals() else []
html_draft = (latest_ui_draft or {}).get('html') or next(
    (call.get('result', {}).get('html') for call in graph_tool_calls
     if call.get('name') == 'generate_marketing_html'),
    None,
)
if html_draft:
    display(HTML(f'<iframe title="Marketing HTML preview" sandbox="" style="width:100%;height:420px;border:1px solid #d9e2f3;border-radius:8px;background:white" srcdoc="{escape(html_draft, quote=True)}"></iframe>'))
else:
    print('같은 노트북 커널에서 UI 또는 Graph 실행으로 생성된 Marketing HTML이 없습니다.')


/Users/I781176/Library/Python/3.14/lib/python/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>01과 같은 Graph 실행과 Agent별 스트리밍 카드를 확인한다.</li><li>Tool을 선택한 Agent의 완료 행과 카드 상세를 확인한다.</li><li>외부 쓰기 Tool이 없음을 확인한다.</li></ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">노트북은 구현을 복사하지 않고 공개 코어 API만 호출합니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))
from src.agent_loop import list_food_mcp_tools, run_food_mcp_loop


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Agent별 Tool 계약 확인</h2><p style="margin: 7px 0;">기존 Graph의 각 전문 Agent에 할당된 Function, API, Agent-as-a-Tool 계약을 공개 API로 확인합니다.</p></div></div>


In [ ]:
list_food_mcp_tools()


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. Graph 실행과 Tool Calling trace</h2><p style="margin: 7px 0;">결과를 다시 계산하지 않고 공개 API가 반환한 Agent 결과, 선택된 Tool, trace를 확인합니다.</p></div></div>


In [ ]:
result = run_food_mcp_loop('고단백 닭가슴살 도시락 출시를 검토해줘.')
print(result['result'])
print('선택된 Tool:', result['tool_calls'])
for event in result['trace']:
    print(event)
